# v5无旧上限五会话同状态HOLD/SOXL/SOXX数值标签

冻结计划先于任何标签结果。覆盖2018–2022逐层点时父本的全部决策前合法分钟意向；填单整股、价格路径完整和五会话成熟只决定事后标签状态。两臂各自原价整股Tiered、0bp额外滑点，事件后HOLD/动作同域锁定五会话，无旧b472/旧上限。发现价值只看2018–2020；2021/2022价值封存于Qlib Recorder。

In [1]:
import os,json,hashlib,warnings,time
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.recorder_run import publish_evidence_manifest
from src.analysis.stage1_capfree_lock_batch import build_capfree_lock_labels
root=Path.cwd();card=root/'research/stage1_v5_capfree_lock_path_labels_20261030';plan_raw=(card/'plan.json').read_bytes();plan=json.loads(plan_raw)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
assert sha(root/plan['vector_module'])==plan['vector_module_sha256']
assert sha(root/plan['support_card']/'summary.json')==plan['support_summary_sha256']
assert sha(root/plan['support_card']/'perday_support.csv')==plan['support_perday_sha256']
assert sha(root/plan['parity_card']/'audit.json')==plan['parity_audit_sha256']
for role in ('parent','frame'):
 s=plan[role];source=root/s['card']
 assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256']
 assert json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
book=read('parent','analysis/native/dated_f696_0')
frame=read('frame','inputs/covered_frame').reset_index(drop=True)
perday=pd.read_csv(root/plan['support_card']/'perday_support.csv',parse_dates=['trade_date','session_close'])
assert len(frame)==len(perday)==plan['expected_sessions'] and book['audit']['status']=='passed' and book['audit']['slippage_bps']==0
print('source identity passed',len(frame))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[29516:MainThread](2026-10-30 14:21:53,298) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[29516:MainThread](2026-10-30 14:21:53,299) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[29516:MainThread](2026-10-30 14:21:53,300) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


source identity passed 1056


In [2]:
started=time.perf_counter()
raw=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$price_valid'],start_time=plan['start']+' 09:31:00',end_time=plan['end']+' 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=raw.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+f.removeprefix('$') for f in x.columns];parts.append(x)
assert parts[0].index.equals(parts[1].index)
quotes=pd.concat(parts,axis=1).sort_index()
assert len(raw)==820440 and quotes.index.is_unique
print('Qlib minute source rows',len(raw))

Qlib minute source rows 820440


In [3]:
# No Qlib Recorder is opened until the complete row/cost/clock gates pass.
labels,mechanics=build_capfree_lock_labels(frame,book,quotes,perday,tariff=json.loads((root/'research/stage1_v5_equal_beta_value_labels_20261030/plan.json').read_text())['tariff'],maximum_intent_rows=plan['budget']['maximum_intent_rows'],maximum_seconds=plan['budget']['maximum_seconds'])
counts=mechanics['counts']
assert counts['intent_rows']==len(labels)==plan['expected_decision_source_intents']
assert counts['mature_path_rows']==plan['expected_mature_path_rows']
assert counts['paired_funded']==plan['expected_paired_funded_rows']
assert labels.duplicated(['trade_date','bar_end_at']).sum()==0
assert labels.loc[labels.status.eq('paired_funded'),'label_available_at'].gt(labels.loc[labels.status.eq('paired_funded'),'decision_at']).all()
assert counts['unknown_path']+counts['terminal_unknown']+counts['mature_path_rows']==counts['intent_rows']
# Discovery-only sign support. Do not inspect held 2021/2022 value columns.
discovery=labels.loc[labels.trade_date.dt.year.le(2020)&labels.status.eq('paired_funded'),['trade_date','relative_value_soxl_minus_soxx','soxl_tail_damage','soxx_tail_damage']].copy()
byday=discovery.groupby('trade_date').agg(value_mean=('relative_value_soxl_minus_soxx','mean'),any_positive=('relative_value_soxl_minus_soxx',lambda x:bool(x.gt(0).any())),any_negative=('relative_value_soxl_minus_soxx',lambda x:bool(x.lt(0).any())),soxl_tail_mean=('soxl_tail_damage','mean'),soxx_tail_mean=('soxx_tail_damage','mean')).reset_index()
byday['year']=byday.trade_date.dt.year
annual=byday.groupby('year').agg(days=('trade_date','size'),positive_days=('any_positive','sum'),negative_days=('any_negative','sum'),day_mean_relative_value=('value_mean','mean'),day_mean_soxl_tail_damage=('soxl_tail_mean','mean'),day_mean_soxx_tail_damage=('soxx_tail_mean','mean')).reset_index()
annual_list=[{'year':int(x.year),'days':int(x.days),'positive_days':int(x.positive_days),'negative_days':int(x.negative_days),'day_mean_relative_value':float(x.day_mean_relative_value),'day_mean_soxl_tail_damage':float(x.day_mean_soxl_tail_damage),'day_mean_soxx_tail_damage':float(x.day_mean_soxx_tail_damage)} for x in annual.itertuples(index=False)]
gate=plan['discovery_sign_support_gate'];years={x['year']:x for x in annual_list}
passed=bool(all(years[y]['positive_days']>=gate[f'minimum_{y}_positive_days'] and years[y]['negative_days']>=gate[f'minimum_{y}_negative_days'] for y in (2019,2020)))
full_elapsed=time.perf_counter()-started
if full_elapsed>plan['budget']['maximum_seconds']:
 raise ValueError('blocked: full label wall-time budget exceeded before Recorder')
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'batch_code_sha256':sha(root/'src/analysis/stage1_capfree_lock_batch.py'),'vector_module_sha256':plan['vector_module_sha256'],'counts':counts,'opening_peak_max_error':mechanics['opening_peak_max_error'],'qlib_minute_rows':len(raw),'batch_elapsed_seconds':mechanics['elapsed_seconds'],'full_pre_recorder_elapsed_seconds':full_elapsed,'discovery_annual':annual_list,'discovery_sign_support_pass':passed,'held_2021_2022_value_columns_inspected':False,'historical_research_passed':False,'decision':'five_session_path_labels_source_and_discovery_sign_pass_information_unproven' if passed else 'five_session_path_labels_discovery_sign_support_failed'}
print(json.dumps(audit,ensure_ascii=False,indent=2))

label_progress 0 367 0.3


label_progress 100 36707 29.9


label_progress 200 73227 60.6


label_progress 300 109567 91.6


label_progress 400 146267 122.8


...

label_progress 700 256007 222.1


label_progress 800 292527 256.1


label_progress 900 329227 291.2


label_progress 1000 365927 326.4


{
  "experiment": "stage1_v5_capfree_lock_path_labels_20261030",
  "plan_sha256": "2bb3dbd2a1b47cc59bf3670be71d75a93bda05cbea26c5f9e5ad81c0a246f2bb",
  "batch_code_sha256": "a1a0b787e01eabfcb9f8f533d9d2359a66ed4d2059880c0bfbff151c33c54987",
  "vector_module_sha256": "aa1be2b7a9ece7e094360cf06e509d9cf28bf8ee456d33d44f01996fc48363b9",
  "counts": {
    "intent_rows": 385932,
    "mature_path_rows": 382996,
    "soxl_funded": 382489,
    "soxx_funded": 332757,
    "paired_funded": 332277,
    "unknown_path": 1468,
    "terminal_unknown": 1468,
    "no_exact_whole_share_pair": 50719
  },
  "opening_peak_max_error": 4.440892098500626e-16,
  "qlib_minute_rows": 820440,
  "batch_elapsed_seconds": 345.8058930829866,
  "full_pre_recorder_elapsed_seconds": 351.0692920840229,
  "discovery_annual": [
    {
      "year": 2018,
      "days": 53,
      "positive_days": 36,
      "negative_days": 51,
      "day_mean_relative_value": -0.0001487213932741548,
      "day_mean_soxl_tail_damage": 0.00429790

In [4]:
# Recorder-first delivery. Large minute rows remain inside Qlib Recorder.
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path='inputs',plan=plan,source_summary={'parent':plan['parent']['recorder_id'],'frame':plan['frame']['recorder_id'],'provider_manifest_sha256':plan['provider_manifest_sha256']})
 recorder.save_objects(artifact_path='sources',batch_builder=(root/'src/analysis/stage1_capfree_lock_batch.py').read_text(),vectorized_path=(root/plan['vector_module']).read_text())
 recorder.save_objects(artifact_path='analysis',labels=labels,discovery_annual=annual,audit=audit)
 recorder.log_metrics(intent_rows=float(counts['intent_rows']),paired_funded_rows=float(counts['paired_funded']),source_gate_pass=1.,discovery_sign_support_pass=float(passed))
assert recorder.status=='FINISHED'
local=tracking/str(recorder.experiment_id)/recorder.id
recorder_size=sum(x.stat().st_size for x in local.rglob('*') if x.is_file())
if recorder_size>plan['budget']['maximum_recorder_bytes']:
 raise ValueError('blocked: Recorder exceeded frozen byte budget; preserve local object')
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':audit['decision'],'historical_research_passed':False,'recorder_id':recorder.id,'experiment_id':str(recorder.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(local),'recorder_bytes':recorder_size,'source_audit':{k:audit[k] for k in ('counts','opening_peak_max_error','qlib_minute_rows','batch_elapsed_seconds','discovery_sign_support_pass')},'discovery_annual':annual_list,'new_model_fits':0,'new_native_candidate_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']},'limitations':['Five-session event/HOLD local path labels, not a trained continuous allocator or complete SOXX candidate account.','2021/2022 values stored but not inspected for selection; overlapping minute events require day grouping and five-session purge.','Historical minute first receipt, actual broker fills and adjusted-price separate dividend receivables remain unqualified.']}
(card/'workflow.yaml').write_text('experiment: '+card.name+'\nnotebook: '+card.name+'.ipynb\nrecorder_uri: '+str(tracking)+'\nartifact_policy: slim_card_recorder_first\n')
publish_evidence_manifest(card,summary)
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'recorder_id':recorder.id,'recorder_bytes':recorder_size,'rows':len(labels),'paired':counts['paired_funded']},ensure_ascii=False))

[29516:MainThread](2026-10-30 14:27:44,626) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_v5_capfree_lock_path_labels_20261030.


[29516:MainThread](2026-10-30 14:27:44,628) INFO - qlib.workflow - [exp.py:258] - Experiment 294476149756418685 starts running ...


[29516:MainThread](2026-10-30 14:27:44,718) INFO - qlib.workflow - [recorder.py:345] - Recorder 2da04a28956b4fb39d07954824ee457d starts running under Experiment 294476149756418685 ...


[29516:MainThread](2026-10-30 14:27:46,029) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{"recorder_id": "2da04a28956b4fb39d07954824ee457d", "recorder_bytes": 64619375, "rows": 385932, "paired": 332277}


## 中文结论

SOXX历史门未通过：0新模型、0新候选账户。Qlib Recorder `2da04a28956b4fb39d07954824ee457d` 保存385932条完整分钟意向、382996条成熟五会话路径、332277条双臂指定整股标签；开盘峰值误差4.44e−16，整批约346秒，NAS档案/所选对象远端读回通过。2018–2020按日等权SOXL−SOXX局部均值−1.487/−0.443/−1.052bp，手续费贡献仅+0.009/+0.009/+0.006bp，其余价格/股数路径主导负均值；2019/2020正负日支持门通过，不等于信号可预测。2021/2022价值封存未用于特征选择。下步冻结分钟自身回撤相对状态/普通分钟控制的信息门，若通过再设计连续目标及完整SOXX 0/2/5bp账户。详情见`conclusion.md`、`discovery_fee_price_attribution.json`与Recorder。